# ARVI — Stratégie `u_ones` : U-Ones (incertain → suspected_opacity)

Notebook autonome. Entraîne un ResNet18 sur CheXpert Small
avec la stratégie **u_ones** et exporte le modèle en `.onnx`.

## Instructions
1. **+ Add Input** → cherche `chexpert` → ajoute `CheXpert-v1.0-small`
2. **Settings → Accelerator → GPU T4 x1** (pas P100 !)
3. **Run All** — durée estimée : ~2h30
4. Dès que tu vois le message SAVE VERSION en bas → clique immédiatement
5. Télécharge `run_summary_u_ones.json` et `arvi_cxr_classifier_u_ones.onnx`
   depuis l'onglet Output et envoie-les à Florielle

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxscript"], check=True)
print("onnxscript OK")

In [ ]:
import os, glob, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score, accuracy_score)
warnings.filterwarnings("ignore")

## 1. Configuration

In [ ]:
STRATEGY    = "u_ones"
LABEL_NAMES = {0: 'normal', 1: 'suspected_opacity'}
LABEL_TO_ID = {v: k for k, v in LABEL_NAMES.items()}
CLASS_ORDER  = [LABEL_NAMES[i] for i in range(len(LABEL_NAMES))]

CONFIG = {
    "seed": 42, "img_size": 224, "batch_size": 64, "num_epochs": 4,
    "learning_rate": 3e-4, "weight_decay": 1e-4, "model_name": "resnet18",
    "num_workers": 0, "dev_split_fraction": 0.10,
    "debug_sample_frac": None,
    "early_stopping_patience": 3,
    "output_dir": "/kaggle/working",
    "evaluate_on_final": True,
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Stratégie : {STRATEGY} | Classes : {CLASS_ORDER} | Device : {device}")
if device.type != "cuda":
    print("ATTENTION : pas de GPU. Settings → Accelerator → GPU T4 x1")

## 2. Localisation du dataset CheXpert

In [ ]:
def find_chexpert():
    for root, dirs, files in os.walk("/kaggle/input"):
        if root[len("/kaggle/input"):].count(os.sep) >= 4:
            dirs[:] = []; continue
        if "train.csv" in files:
            tc = os.path.join(root, "train.csv")
            vc = os.path.join(root, "valid.csv")
            dirs[:] = []
            return tc, vc if os.path.isfile(vc) else None, root
    raise FileNotFoundError("train.csv introuvable. Ajoute CheXpert via + Add Input.")

def resolve_path(raw, root):
    parts = raw.split("/", 1)
    cands = [raw, os.path.join(root, raw)]
    if len(parts) == 2:
        cands += [os.path.join(root, parts[1]),
                  os.path.join(os.path.dirname(root), parts[1])]
    for c in cands:
        if os.path.isfile(c): return c
    raise FileNotFoundError(f"Image introuvable : {raw}")

TRAIN_CSV, VALID_CSV, DATA_ROOT = find_chexpert()
print(f"train.csv : {TRAIN_CSV}\nvalid.csv : {VALID_CSV}")

## 3. Construction des labels

In [ ]:
def mapping_fn(value):
    if pd.isna(value): return "normal"
    v = float(value)
    if v == 0.0: return "normal"
    return "suspected_opacity"  # 1.0 et -1.0

def build_df(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df["Frontal/Lateral"] == "Frontal"].copy()
    df["label_name"] = df["Lung Opacity"].apply(mapping_fn)
    df["label_id"]   = df["label_name"].map(LABEL_TO_ID)
    df["patient_id"] = df["Path"].apply(
        lambda p: next((x for x in p.split("/") if x.startswith("patient")), p))
    df["was_originally_uncertain"] = df["Lung Opacity"].apply(
        lambda v: pd.isna(v) or float(v) == -1.0)
    return df.reset_index(drop=True)

def maybe_sub(df, frac, seed):
    if frac is None: return df
    return df.groupby("label_name", group_keys=False).apply(
        lambda g: g.sample(frac=frac, random_state=seed)).reset_index(drop=True)

full_df = maybe_sub(build_df(TRAIN_CSV), CONFIG["debug_sample_frac"], CONFIG["seed"])
print(f"Images train : {len(full_df)}")
print(full_df["label_name"].value_counts().to_dict())

gss = GroupShuffleSplit(1, test_size=CONFIG["dev_split_fraction"], random_state=CONFIG["seed"])
tr_idx, dv_idx = next(gss.split(full_df, groups=full_df["patient_id"]))
train_df = full_df.iloc[tr_idx].reset_index(drop=True)
dev_df   = full_df.iloc[dv_idx].reset_index(drop=True)
assert not (set(train_df["patient_id"]) & set(dev_df["patient_id"]))

final_df = None
if VALID_CSV:
    final_df = build_df(VALID_CSV)
    assert not ((set(train_df["patient_id"]) | set(dev_df["patient_id"])) & set(final_df["patient_id"]))

print(f"train={len(train_df)} | dev={len(dev_df)} | final={len(final_df) if final_df is not None else 0}")

## 4. Dataset PyTorch

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

class CXRDataset(Dataset):
    def __init__(self, df, root, tf):
        self.df, self.root, self.tf = df.reset_index(drop=True), root, tf
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(resolve_path(row["Path"], self.root)).convert("RGB")
        return self.tf(img), int(row["label_id"]), row["Path"], bool(row["was_originally_uncertain"])

train_tf = transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf  = transforms.Compose([transforms.Resize((224,224)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

kw = dict(num_workers=0, pin_memory=True)
train_loader = DataLoader(CXRDataset(train_df, DATA_ROOT, train_tf),
    batch_size=CONFIG["batch_size"], shuffle=True,  drop_last=True, **kw)
dev_loader   = DataLoader(CXRDataset(dev_df,   DATA_ROOT, eval_tf),
    batch_size=CONFIG["batch_size"], shuffle=False, **kw)
final_loader = DataLoader(CXRDataset(final_df, DATA_ROOT, eval_tf),
    batch_size=CONFIG["batch_size"], shuffle=False, **kw) if final_df is not None else None

_ = CXRDataset(train_df, DATA_ROOT, train_tf)[0]
print("Sanity check OK")

## 5. Modèle

In [ ]:
def build_model(n):
    try:    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except: m = models.resnet18(pretrained=True)
    m.fc = nn.Linear(m.fc.in_features, n)
    return m

def class_weights_fn(df, n):
    c = df["label_id"].value_counts().reindex(range(n), fill_value=0)
    return torch.tensor((c.sum()/(n*c.clip(lower=1))).values, dtype=torch.float32)

model     = build_model(len(LABEL_NAMES)).to(device)
cweights  = class_weights_fn(train_df, len(LABEL_NAMES)).to(device)
criterion = nn.CrossEntropyLoss(weight=cweights)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scaler    = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
print(f"Poids classes : {dict(zip(CLASS_ORDER, cweights.tolist()))}")

## 6. Entraînement

In [ ]:
def train_epoch(model, loader, opt, crit, dev, scaler):
    model.train(); ls = 0
    for imgs, labels, _, _ in tqdm(loader, desc="train", leave=False):
        imgs, labels = imgs.to(dev, non_blocking=True), labels.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=dev.type, enabled=(dev.type=="cuda")):
            out = model(imgs); loss = crit(out, labels)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ls += loss.item() * imgs.size(0)
    return ls / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, crit, dev):
    model.eval()
    preds, labels, confs, paths, uncs = [], [], [], [], []
    ls = 0
    for imgs, lbs, pts, unc in tqdm(loader, desc="eval", leave=False):
        imgs, lbs_gpu = imgs.to(dev), lbs.to(dev)
        out  = model(imgs); loss = crit(out, lbs_gpu); ls += loss.item()*imgs.size(0)
        pr   = torch.softmax(out, 1); cf, pd_ = pr.max(1)
        preds.extend(pd_.cpu().tolist()); labels.extend(lbs.tolist())
        confs.extend(cf.cpu().tolist());  paths.extend(pts)
        uncs.extend([bool(x) for x in unc.tolist()])
    return dict(loss=ls/len(loader.dataset),
                accuracy=accuracy_score(labels, preds),
                macro_f1=f1_score(labels, preds, average="macro"),
                preds=preds, labels=labels,
                confidences=confs, paths=paths, originally_uncertain=uncs)

In [ ]:
best_f1  = -1.0
best_pt  = os.path.join(CONFIG["output_dir"], f"best_model_{STRATEGY}.pt")
history  = {"train_loss":[], "dev_loss":[], "dev_macro_f1":[]}
no_impr  = 0

for epoch in range(1, CONFIG["num_epochs"]+1):
    t0  = time.time()
    tl  = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
    dm  = evaluate(model, dev_loader, criterion, device)
    history["train_loss"].append(tl)
    history["dev_loss"].append(dm["loss"])
    history["dev_macro_f1"].append(dm["macro_f1"])
    print(f"[Epoch {epoch}/{CONFIG['num_epochs']}] loss={tl:.4f} dev_macro_f1={dm['macro_f1']:.4f} ({time.time()-t0:.0f}s)")
    if dm["macro_f1"] > best_f1:
        best_f1 = dm["macro_f1"]; no_impr = 0
        torch.save(model.state_dict(), best_pt)
        print(f"  -> meilleur modèle sauvegardé (F1={best_f1:.4f})")
    else:
        no_impr += 1
        if no_impr >= CONFIG["early_stopping_patience"]:
            print("Early stopping."); break

print(f"\nMeilleur dev macro-F1 : {best_f1:.4f}")

## 7. Évaluation

In [ ]:
model.load_state_dict(torch.load(best_pt, map_location=device))
model.eval()
dev_m   = evaluate(model, dev_loader, criterion, device)
dev_rep = classification_report(dev_m["labels"], dev_m["preds"],
                                 target_names=CLASS_ORDER, digits=3, output_dict=True)
dev_cm  = confusion_matrix(dev_m["labels"], dev_m["preds"])
print(classification_report(dev_m["labels"], dev_m["preds"], target_names=CLASS_ORDER, digits=3))

ConfusionMatrixDisplay(confusion_matrix=dev_cm, display_labels=CLASS_ORDER).plot(
    cmap="Blues", values_format="d")
plt.title(f"Confusion matrix (dev) — {STRATEGY}")
plt.tight_layout(); plt.show()

final_rep, final_cm, final_m = None, None, None
if CONFIG["evaluate_on_final"] and final_loader is not None:
    final_m   = evaluate(model, final_loader, criterion, device)
    final_rep = classification_report(final_m["labels"], final_m["preds"],
                                       target_names=CLASS_ORDER, digits=3, output_dict=True)
    final_cm  = confusion_matrix(final_m["labels"], final_m["preds"])
    print("=== Split final (radiologues) ===")
    print(classification_report(final_m["labels"], final_m["preds"], target_names=CLASS_ORDER, digits=3))

## 8. Export ONNX

In [ ]:
model.eval()
dummy     = torch.randn(1, 3, 224, 224, device=device)
onnx_path = os.path.join(CONFIG["output_dir"], f"arvi_cxr_classifier_{STRATEGY}.onnx")
kw_onnx   = dict(input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input":{0:"batch"}, "logits":{0:"batch"}})
try:
    torch.onnx.export(model, dummy, onnx_path, opset_version=17, **kw_onnx)
except Exception as e:
    print(f"opset 17 échoué ({e}), retry opset 13")
    torch.onnx.export(model, dummy, onnx_path, opset_version=13, **kw_onnx)

try:
    import onnx; onnx.checker.check_model(onnx.load(onnx_path))
    print(f"ONNX OK : {onnx_path}")
except ImportError:
    print("Module onnx absent, vérification ignorée.")

## 9. Sauvegarde du résumé

In [ ]:
def summarize_uncertain(m, lnames):
    idx = [i for i, f in enumerate(m["originally_uncertain"]) if f]
    if not idx: return {"n_cases": 0}
    pn  = [lnames[m["preds"][i]] for i in idx]
    res = {"n_cases": len(idx),
            "predicted_label_distribution": pd.Series(pn).value_counts().to_dict(),
            "mean_confidence": float(np.mean([m["confidences"][i] for i in idx]))}
    if "uncertain" in lnames.values():
        res["pct_correctly_flagged_uncertain"] = round(100*sum(1 for n in pn if n=="uncertain")/len(idx), 1)
    return res

dev_unc   = summarize_uncertain(dev_m, LABEL_NAMES)
final_unc = summarize_uncertain(final_m, LABEL_NAMES) if final_m else {"n_cases": 0}

run_summary = {
    "strategy":                         STRATEGY,
    "label_names":                      LABEL_NAMES,
    "class_order":                      CLASS_ORDER,
    "best_dev_macro_f1":                best_f1,
    "dev_classification_report":        dev_rep,
    "dev_confusion_matrix":             dev_cm.tolist(),
    "dev_originally_uncertain_summary": dev_unc,
    "final_classification_report":      final_rep,
    "final_confusion_matrix":           final_cm.tolist() if final_cm is not None else None,
    "final_originally_uncertain_summary": final_unc,
    "elapsed_seconds":                  0,
    "epochs_trained":                   len(history["train_loss"]),
    "history":                          history,
}

summary_path = os.path.join(CONFIG["output_dir"], f"run_summary_{STRATEGY}.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2, default=str)

with open(os.path.join(CONFIG["output_dir"], f"model_card_{STRATEGY}.json"), "w") as f:
    json.dump({
        "strategy": STRATEGY, "model": CONFIG["model_name"],
        "labels": LABEL_NAMES,
        "input": {"shape":[1,3,224,224], "mean":[0.485,0.456,0.406], "std":[0.229,0.224,0.225]},
        "output": {"name":"logits","postprocessing":"softmax + argmax"},
        "best_dev_macro_f1": best_f1,
        "warning": "Prototype pédagogique. Non destiné au diagnostic clinique.",
    }, f, ensure_ascii=False, indent=2)

print(f"Résumé sauvegardé : {summary_path}")

## Résumé final

In [ ]:
print("="*60)
print(f"STRATÉGIE       : {STRATEGY}")
print(f"Meilleur F1 dev : {best_f1:.4f}")
print(f"Époques         : {len(history['train_loss'])}")
print("Fichiers produits :")
for fname in [f"best_model_{STRATEGY}.pt",
              f"arvi_cxr_classifier_{STRATEGY}.onnx",
              f"model_card_{STRATEGY}.json",
              f"run_summary_{STRATEGY}.json"]:
    fpath = os.path.join(CONFIG["output_dir"], fname)
    ok    = os.path.isfile(fpath)
    size  = f"({os.path.getsize(fpath)/1024/1024:.1f} Mo)" if ok else ""
    print(f"  {'OK' if ok else 'MANQUANT'} {fname} {size}")
print("="*60)
print()
print("*"*60)
print("*  FAIS SAVE VERSION MAINTENANT (bouton en haut à droite)  *")
print("*  puis télécharge run_summary et .onnx depuis Output      *")
print("*"*60)